# Calculate VIMFC and MFC (850 hPa) for LBM Experiments

This notebook computes:
1. Vertically Integrated Moisture Flux (VIMF) and its Convergence (VIMFC).
2. Moisture Flux at 850 hPa (MF_850) and its Convergence (MFC_850).

Experiments processed:
- : P1 basic state (1981–2006) and P1 forcing.
- : P2 basic state (2007–2025) and P2 forcing.
- : Difference (EXP_B_S-123 − CTRL).

The results are saved to  for decoupled plotting.


In [1]:
import os
from pathlib import Path
import numpy as np
import xarray as xr
import geocat.comp as gc

# Paths
NOTEBOOK_DIR = Path('/Users/rizzie/Work/PaperENSO/Scripts_v2/Supplement/ConvergenceModel')
LBM_PATH = Path('/Users/rizzie/Work/PaperENSO/Scripts_v2/Fig5/combined_ncep_r2_moist_lbm.nc')
BS_PATH = Path('/Users/rizzie/LinearBaroclinicModel/make_basic_state/output/basic_state_ncep_r2_era5sst_djf_all_periods.nc')
OUTPUT_NC_PATH = NOTEBOOK_DIR / 'convergence_vimfc_850.nc'

print(f'LBM file exists: {LBM_PATH.exists()}')
print(f'BS file exists:  {BS_PATH.exists()}')


LBM file exists: True
BS file exists:  True


In [2]:
# Load datasets
ds_lbm = xr.open_dataset(LBM_PATH, decode_times=False)
ds_bs = xr.open_dataset(BS_PATH, decode_times=False)

# Align BS latitude (south-to-north) to LBM latitude (north-to-south)
ds_bs = ds_bs.isel(lat=slice(None, None, -1)).assign_coords(lat=ds_lbm.lat)

# Constants & parameters
g = 9.80665                 # m s^-2
time_idx = 14               # 15th time step (Day 15 / t = 336.0 h)
lev_Pa = ds_lbm.lev * 100.0 # Convert hPa to Pa

# Experiment configuration
# period 1: 1981-2006 (P1), period 2: 2007-2025 (P2)
exp_configs = {
    'CTRL': {'exp_id': 'CTRL', 'period_idx': 1},
    'EXP_B_S-123': {'exp_id': 'EXP_B_S-123', 'period_idx': 2}
}

print(f'Level (Pa): {lev_Pa.values}')
print(f'Selected time step: {time_idx} (t = {ds_lbm.time.values[time_idx]} h)')


Level (Pa): [100000.  95000.  90000.  85000.  70000.  60000.  50000.  40000.  30000.
  25000.  20000.  15000.  10000.   7000.   5000.   3000.   2000.   1000.
    700.    500.]
Selected time step: 14 (t = 336.0 h)


In [3]:
results = {}

for name, cfg in exp_configs.items():
    exp_id = cfg['exp_id']
    pidx = cfg['period_idx']
    
    # 1. Surface pressure and vertical layer thickness (dp)
    psbar = ds_bs['p'].isel(period=pidx)
    ps_prime = ds_lbm['p'].sel(experiment=exp_id).isel(time=time_idx).squeeze('lev_2')
    ps_total_Pa = (psbar + ps_prime) * 100.0  # hPa to Pa
    
    dp_raw = gc.delta_pressure(lev_Pa, ps_total_Pa)
    dp = xr.DataArray(
        dp_raw,
        coords={'lat': ds_lbm.lat, 'lon': ds_lbm.lon, 'lev': lev_Pa},
        dims=['lat', 'lon', 'lev']
    )
    
    # 2. Extract wind and moisture fields
    u_prime = ds_lbm['u'].sel(experiment=exp_id).isel(time=time_idx).assign_coords(lev=lev_Pa)
    v_prime = ds_lbm['v'].sel(experiment=exp_id).isel(time=time_idx).assign_coords(lev=lev_Pa)
    q_prime = ds_lbm['q'].sel(experiment=exp_id).isel(time=time_idx).assign_coords(lev=lev_Pa)
    
    ubar = ds_bs['u'].isel(period=pidx).rename({'sigma': 'lev'}).assign_coords(lev=lev_Pa)
    vbar = ds_bs['v'].isel(period=pidx).rename({'sigma': 'lev'}).assign_coords(lev=lev_Pa)
    qbar = ds_bs['q'].isel(period=pidx).rename({'sigma': 'lev'}).assign_coords(lev=lev_Pa)
    
    # 3. Vertically Integrated Moisture Flux (VIMF) and Convergence (VIMFC)
    u_q = (qbar * u_prime + q_prime * ubar) * dp / g
    v_q = (qbar * v_prime + q_prime * vbar) * dp / g
    
    VIMF_u = u_q.sum(dim='lev')
    VIMF_v = v_q.sum(dim='lev')
    
    grads_u = gc.gradient(VIMF_u)
    grads_v = gc.gradient(VIMF_v)
    VIMFC = -(grads_u[1] + grads_v[0])
    VIMFC = xr.DataArray(VIMFC.values, coords=[('lat', ds_lbm.lat.values), ('lon', ds_lbm.lon.values)])
    
    # 4. Moisture Flux at 850 hPa (MF_850) and Convergence (MFC_850)
    # 850 hPa corresponds to 85000 Pa
    u_p_850 = u_prime.sel(lev=85000.0)
    v_p_850 = v_prime.sel(lev=85000.0)
    q_p_850 = q_prime.sel(lev=85000.0)
    
    ubar_850 = ubar.sel(lev=85000.0)
    vbar_850 = vbar.sel(lev=85000.0)
    qbar_850 = qbar.sel(lev=85000.0)
    
    MF_850_u = qbar_850 * u_p_850 + q_p_850 * ubar_850
    MF_850_v = qbar_850 * v_p_850 + q_p_850 * vbar_850
    
    grads_mf_u = gc.gradient(MF_850_u)
    grads_mf_v = gc.gradient(MF_850_v)
    MFC_850 = -(grads_mf_u[1] + grads_mf_v[0])
    MFC_850 = xr.DataArray(MFC_850.values, coords=[('lat', ds_lbm.lat.values), ('lon', ds_lbm.lon.values)])
    
    results[name] = {
        'VIMF_u': VIMF_u,
        'VIMF_v': VIMF_v,
        'VIMFC': VIMFC,
        'MF_850_u': MF_850_u,
        'MF_850_v': MF_850_v,
        'MFC_850': MFC_850
    }
    print(f'Successfully calculated fields for {name}')

# 5. Compute difference: EXP_B_S-123 - CTRL
diff_name = 'EXP_minus_CTRL'
results[diff_name] = {
    'VIMF_u': results['EXP_B_S-123']['VIMF_u'] - results['CTRL']['VIMF_u'],
    'VIMF_v': results['EXP_B_S-123']['VIMF_v'] - results['CTRL']['VIMF_v'],
    'VIMFC': results['EXP_B_S-123']['VIMFC'] - results['CTRL']['VIMFC'],
    'MF_850_u': results['EXP_B_S-123']['MF_850_u'] - results['CTRL']['MF_850_u'],
    'MF_850_v': results['EXP_B_S-123']['MF_850_v'] - results['CTRL']['MF_850_v'],
    'MFC_850': results['EXP_B_S-123']['MFC_850'] - results['CTRL']['MFC_850'],
}
print(f'Successfully calculated differences ({diff_name})')


Successfully calculated fields for CTRL


Successfully calculated fields for EXP_B_S-123
Successfully calculated differences (EXP_minus_CTRL)


In [4]:
# Build xarray Dataset
cases = ['CTRL', 'EXP_B_S-123', 'EXP_minus_CTRL']
case_coord = xr.DataArray(np.array(cases, dtype='S16'), dims='case', name='case')

def clean_da(da):
    return da.reset_coords(drop=True)

out_ds = xr.Dataset(
    {
        'VIMF_u': xr.concat([clean_da(results[c]['VIMF_u']) for c in cases], dim=case_coord),
        'VIMF_v': xr.concat([clean_da(results[c]['VIMF_v']) for c in cases], dim=case_coord),
        'VIMFC': xr.concat([clean_da(results[c]['VIMFC']) for c in cases], dim=case_coord),
        'MF_850_u': xr.concat([clean_da(results[c]['MF_850_u']) for c in cases], dim=case_coord),
        'MF_850_v': xr.concat([clean_da(results[c]['MF_850_v']) for c in cases], dim=case_coord),
        'MFC_850': xr.concat([clean_da(results[c]['MFC_850']) for c in cases], dim=case_coord),
    },
    coords={
        'lat': ds_lbm.lat.values,
        'lon': ds_lbm.lon.values,
    },
    attrs={
        'title': 'VIMFC and 850-hPa MFC for LBM Experiments',
        'description': 'Computed from combined_ncep_r2_moist_lbm.nc and basic_state_ncep_r2_era5sst_djf_all_periods.nc',
        'time_index': str(time_idx),
        'time_hours': str(ds_lbm.time.values[time_idx]),
        'cases': ', '.join(cases),
    }
)

# Variable attributes
out_ds['VIMF_u'].attrs = {'long_name': 'Zonal vertically integrated moisture flux', 'units': 'kg m^-1 s^-1'}
out_ds['VIMF_v'].attrs = {'long_name': 'Meridional vertically integrated moisture flux', 'units': 'kg m^-1 s^-1'}
out_ds['VIMFC'].attrs = {'long_name': 'Vertically integrated moisture flux convergence', 'units': 'kg m^-2 s^-1'}
out_ds['MF_850_u'].attrs = {'long_name': 'Zonal moisture flux at 850 hPa', 'units': 'kg kg^-1 m s^-1'}
out_ds['MF_850_v'].attrs = {'long_name': 'Meridional moisture flux at 850 hPa', 'units': 'kg kg^-1 m s^-1'}
out_ds['MFC_850'].attrs = {'long_name': 'Moisture flux convergence at 850 hPa', 'units': 'kg kg^-1 s^-1'}

out_ds.to_netcdf(OUTPUT_NC_PATH)
print(f'Successfully saved results to {OUTPUT_NC_PATH}')
print(out_ds)


Successfully saved results to /Users/rizzie/Work/PaperENSO/Scripts_v2/Supplement/ConvergenceModel/convergence_vimfc_850.nc
<xarray.Dataset> Size: 247kB
Dimensions:   (lon: 64, lat: 32, case: 3)
Coordinates:
  * lon       (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
  * lat       (lat) float64 256B 85.76 80.27 74.75 ... -74.75 -80.27 -85.76
  * case      (case) |S16 48B b'CTRL' b'EXP_B_S-123' b'EXP_minus_CTRL'
Data variables:
    VIMF_u    (case, lat, lon) float64 49kB -0.6528 -0.5564 ... -0.008057
    VIMF_v    (case, lat, lon) float64 49kB 0.7801 0.8147 ... 0.03996 0.02476
    VIMFC     (case, lat, lon) float64 49kB -1.21e-06 -1.002e-06 ... 5.929e-08
    MF_850_u  (case, lat, lon) float32 25kB -0.0001282 -0.0001068 ... nan nan
    MF_850_v  (case, lat, lon) float32 25kB 0.0001673 0.0001742 ... nan nan
    MFC_850   (case, lat, lon) float64 49kB -2.484e-10 -1.79e-10 ... nan nan
Attributes:
    title:        VIMFC and 850-hPa MFC for LBM Experiments
    description:  C